# Layer 2 — Findings Summary

Flagging engine output. Ranks findings from the descriptive pass by
significance and narrative relevance.

**What this notebook produces:**
- Correlation flags (strong pairs, sorted by strength)
- Changepoint clusters (multiple metrics shifting near the same year)
- Divergence flags (metrics moving in opposite directions)
- Outlier regions (states/districts that break the pattern)
- Custom hypothesis test results (from config.yaml)

**Action required:** Review the ranked findings below and mark which ones
to pursue in Layer 3 deep-dives.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir

PROJECT_NAME = "qol-immigration"

config = load_config(PROJECT_NAME)
processed_dir = get_data_processed_dir(config)

# Load the three analysis panels
PANELS = ["state_panel", "national_panel", "district_crosssection"]
datasets = {}
for name in PANELS:
    path = processed_dir / f"{name}.parquet"
    if path.exists():
        datasets[name] = pd.read_parquet(path)

# Curate state panel to key indicators (same selection as 01_descriptive)
STATE_INDICATORS = [
    "year", "state_fips", "state_name", "state_abbr",
    "median_household_income", "poverty_pct", "real_gdp_millions",
    "hpi_annual_avg", "life_expectancy",
    "acgr_total", "bachelors_pct",
    "foreign_born_pct", "lpr_per_million", "refugees_per_million",
    "naturalizations_total", "asylees_total",
    "voter_reg_total_pct",
]
if "state_panel" in datasets:
    available = [c for c in STATE_INDICATORS if c in datasets["state_panel"].columns]
    datasets["state_panel"] = datasets["state_panel"][available].copy()

print(f"Loaded {len(datasets)} dataset(s): {list(datasets.keys())}")

Loaded 3 dataset(s): ['state_panel', 'national_panel', 'district_crosssection']


## Correlation Flags

In [2]:
from pipeline.flagging import flag_correlations

CORRELATION_THRESHOLD = 0.5  # |r| above this gets flagged

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    flags = flag_correlations(df, threshold=CORRELATION_THRESHOLD)
    display(flags)


--- state_panel ---


,var_a,var_b,pearson_r,spearman_r,abs_max,flag_type,strength,description
0,real_gdp_millions,naturalizations_total,0.9420,0.9409,0.9420,correlation,strong,real_gdp_millions and naturalizations_total: P...
1,year,hpi_annual_avg,0.8107,0.9214,0.9214,correlation,strong,"year and hpi_annual_avg: Pearson r=0.8107, Spe..."
2,foreign_born_pct,lpr_per_million,0.9171,0.9190,0.9190,correlation,strong,foreign_born_pct and lpr_per_million: Pearson ...
3,naturalizations_total,asylees_total,0.8546,0.8683,0.8683,correlation,strong,naturalizations_total and asylees_total: Pears...
4,median_household_income,bachelors_pct,0.8092,0.8552,0.8552,correlation,strong,median_household_income and bachelors_pct: Pea...
5,real_gdp_millions,asylees_total,0.8289,0.8430,0.8430,correlation,strong,real_gdp_millions and asylees_total: Pearson r...
6,hpi_annual_avg,bachelors_pct,0.7761,0.8059,0.8059,correlation,strong,hpi_annual_avg and bachelors_pct: Pearson r=0....
7,median_household_income,hpi_annual_avg,0.7761,0.7536,0.7761,correlation,moderate,median_household_income and hpi_annual_avg: Pe...
8,poverty_pct,life_expectancy,-0.7361,-0.7178,0.7361,correlation,moderate,poverty_pct and life_expectancy: Pearson r=-0....
9,foreign_born_pct,naturalizations_total,0.7267,0.7279,0.7279,correlation,moderate,foreign_born_pct and naturalizations_total: Pe...



--- national_panel ---


,var_a,var_b,pearson_r,spearman_r,abs_max,flag_type,strength,description
0,median_household_income_real,median_weekly_earnings_real,0.9176,0.8275,0.9176,correlation,strong,median_household_income_real and median_weekly...
1,year,gini_coefficient,0.9161,0.8749,0.9161,correlation,strong,"year and gini_coefficient: Pearson r=0.9161, S..."
2,year,median_household_income_real,0.8821,0.8561,0.8821,correlation,strong,year and median_household_income_real: Pearson...
3,year,median_weekly_earnings_real,0.8650,0.8715,0.8715,correlation,strong,year and median_weekly_earnings_real: Pearson ...
4,gini_coefficient,median_weekly_earnings_real,0.7251,0.8447,0.8447,correlation,strong,gini_coefficient and median_weekly_earnings_re...
5,gini_coefficient,median_household_income_real,0.7821,0.8101,0.8101,correlation,strong,gini_coefficient and median_household_income_r...
6,median_household_income_real,poverty_rate,-0.4862,-0.5449,0.5449,correlation,moderate,median_household_income_real and poverty_rate:...



--- district_crosssection ---


,var_a,var_b,pearson_r,spearman_r,abs_max,flag_type,strength,description
0,district_number,district_number_cis,1.0000,1.0000,1.0000,correlation,strong,district_number and district_number_cis: Pears...
1,Romney %,2012 Pres. Margin,-0.9996,-0.9995,0.9996,correlation,strong,Romney % and 2012 Pres. Margin: Pearson r=-0.9...
2,Trump %,2020 Pres. Margin,-0.9996,-0.9996,0.9996,correlation,strong,Trump % and 2020 Pres. Margin: Pearson r=-0.99...
3,Biden %,2020 Pres. Margin,0.9996,0.9996,0.9996,correlation,strong,Biden % and 2020 Pres. Margin: Pearson r=0.999...
4,Obama %,2012 Pres. Margin,0.9995,0.9995,0.9995,correlation,strong,Obama % and 2012 Pres. Margin: Pearson r=0.999...
...,...,...,...,...,...,...,...,...
216,2016 Pres. Margin,U.S. Citizens 18+ Population,-0.5117,-0.5022,0.5117,correlation,moderate,2016 Pres. Margin and U.S. Citizens 18+ Popula...
217,Clinton %,U.S. Citizens 18+ Population,-0.5110,-0.4934,0.5110,correlation,moderate,Clinton % and U.S. Citizens 18+ Population: Pe...
218,white_cvap_pct,Democratic share of two party vote,-0.5085,-0.5098,0.5098,correlation,moderate,white_cvap_pct and Democratic share of two par...
219,Trump %.1,U.S. Citizens 18+ Population,0.5078,0.5026,0.5078,correlation,moderate,Trump %.1 and U.S. Citizens 18+ Population: Pe...


## Changepoint Clusters

In [3]:
from pipeline.flagging import flag_changepoint_clusters

TIME_COL = "year"  # UPDATE if different

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    clusters = flag_changepoint_clusters(df, time_col=TIME_COL)
    display(clusters)


--- state_panel ---


,cluster_year,num_indicators,indicators,flag_type,strength,description
0,2017,4,"median_household_income, real_gdp_millions, ba...",changepoint_cluster,strong,4 indicators shifted near 2017: median_househo...
1,2015,3,"median_household_income, real_gdp_millions, fo...",changepoint_cluster,strong,3 indicators shifted near 2015: median_househo...
2,2018,3,"real_gdp_millions, hpi_annual_avg, bachelors_pct",changepoint_cluster,strong,3 indicators shifted near 2018: real_gdp_milli...
3,2020,2,"hpi_annual_avg, bachelors_pct",changepoint_cluster,moderate,2 indicators shifted near 2020: hpi_annual_avg...



--- national_panel ---


,cluster_year,num_indicators,indicators,flag_type,strength,description
0,1999,2,"median_household_income_real, median_weekly_ea...",changepoint_cluster,moderate,2 indicators shifted near 1999: median_househo...


## Divergence Flags

In [4]:
from pipeline.flagging import flag_divergences

for name, df in datasets.items():
    if TIME_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    divergences = flag_divergences(df, time_col=TIME_COL, config=config)
    display(divergences)


--- state_panel ---


,rising_indicator,falling_indicator,rising_slope,falling_slope,flag_type,strength,narrative_note,description
0,median_household_income,poverty_pct,1558.038982,-0.066858,divergence,moderate,Opposite trends detected.,median_household_income (rising) vs poverty_pc...
1,median_household_income,lpr_per_million,1558.038982,-41.756680,divergence,moderate,Opposite trends detected.,median_household_income (rising) vs lpr_per_mi...
2,median_household_income,refugees_per_million,1558.038982,-17.379583,divergence,moderate,Opposite trends detected.,median_household_income (rising) vs refugees_p...
3,median_household_income,voter_reg_total_pct,1558.038982,-0.019416,divergence,moderate,Opposite trends detected.,median_household_income (rising) vs voter_reg_...
4,real_gdp_millions,poverty_pct,7012.639314,-0.066858,divergence,moderate,Opposite trends detected.,real_gdp_millions (rising) vs poverty_pct (fal...
5,real_gdp_millions,lpr_per_million,7012.639314,-41.756680,divergence,moderate,Opposite trends detected.,real_gdp_millions (rising) vs lpr_per_million ...
6,real_gdp_millions,refugees_per_million,7012.639314,-17.379583,divergence,moderate,Opposite trends detected.,real_gdp_millions (rising) vs refugees_per_mil...
7,real_gdp_millions,voter_reg_total_pct,7012.639314,-0.019416,divergence,moderate,Opposite trends detected.,real_gdp_millions (rising) vs voter_reg_total_...
8,hpi_annual_avg,poverty_pct,10.227203,-0.066858,divergence,moderate,Opposite trends detected.,hpi_annual_avg (rising) vs poverty_pct (falling)
9,hpi_annual_avg,lpr_per_million,10.227203,-41.756680,divergence,moderate,Opposite trends detected.,hpi_annual_avg (rising) vs lpr_per_million (fa...



--- national_panel ---


,note
0,No divergences found.


## Outlier Regions

In [5]:
from pipeline.flagging import flag_outlier_regions

GEO_COL = "state_name"

for name, df in datasets.items():
    if GEO_COL not in df.columns:
        continue
    print(f"\n--- {name} ---")
    outliers = flag_outlier_regions(df, geo_col=GEO_COL)
    display(outliers)


--- state_panel ---


,indicator,region,value,z_score,direction,flag_type,strength,description
0,asylees_total,California,8.475454e+03,5.6159,high,outlier_region,strong,California is a high outlier on asylees_total ...
1,naturalizations_total,California,1.548309e+05,4.9560,high,outlier_region,strong,California is a high outlier on naturalization...
2,real_gdp_millions,California,2.334077e+06,4.7308,high,outlier_region,strong,California is a high outlier on real_gdp_milli...
3,bachelors_pct,District of Columbia,5.651130e+01,4.1350,high,outlier_region,strong,District of Columbia is a high outlier on bach...
4,asylees_total,New York,5.608182e+03,3.5864,high,outlier_region,strong,New York is a high outlier on asylees_total (z...
5,acgr_total,District of Columbia,6.711110e+01,-3.2543,low,outlier_region,strong,District of Columbia is a low outlier on acgr_...
6,hpi_annual_avg,Massachusetts,4.859186e+02,3.0804,high,outlier_region,strong,Massachusetts is a high outlier on hpi_annual_...
7,foreign_born_pct,California,2.703170e+01,2.9465,high,outlier_region,moderate,California is a high outlier on foreign_born_p...
8,lpr_per_million,New York,6.290268e+03,2.9329,high,outlier_region,moderate,New York is a high outlier on lpr_per_million ...
9,voter_reg_total_pct,North Dakota,8.430770e+01,2.7230,high,outlier_region,moderate,North Dakota is a high outlier on voter_reg_to...



--- district_crosssection ---


,indicator,region,value,z_score,direction,flag_type,strength,description
0,asian_cvap_pct,Hawaii,3.900000e+01,5.7571,high,outlier_region,strong,Hawaii is a high outlier on asian_cvap_pct (z=...
1,district_number_cis,California,2.650000e+01,4.4001,high,outlier_region,strong,California is a high outlier on district_numbe...
2,district_number,California,2.650000e+01,4.3620,high,outlier_region,strong,California is a high outlier on district_numbe...
3,hispanic_cvap_pct,New Mexico,4.156670e+01,3.6053,high,outlier_region,strong,New Mexico is a high outlier on hispanic_cvap_...
4,No. of Republican Votes,South Dakota,2.538210e+05,3.5664,high,outlier_region,strong,South Dakota is a high outlier on No. of Repub...
5,Total 18+ Population,Delaware,8.101690e+05,3.4861,high,outlier_region,strong,Delaware is a high outlier on Total 18+ Popula...
6,U.S. Citizens 18+ Population,Delaware,7.697490e+05,3.3994,high,outlier_region,strong,Delaware is a high outlier on U.S. Citizens 18...
7,Total Population,Delaware,1.018396e+06,3.1472,high,outlier_region,strong,Delaware is a high outlier on Total Population...
8,white_cvap_pct,Hawaii,2.605000e+01,-3.1052,low,outlier_region,strong,Hawaii is a low outlier on white_cvap_pct (z=-...
9,district_number_cis,Texas,1.950000e+01,2.9854,high,outlier_region,moderate,Texas is a high outlier on district_number_cis...


## Custom Hypothesis Tests

In [6]:
from pipeline.flagging import test_hypotheses

hypotheses = config.get("hypotheses", [])
if not hypotheses:
    print("No custom hypotheses defined in config.yaml.")
else:
    results = test_hypotheses(datasets, hypotheses, config)
    for r in results:
        print(f"\n--- {r['name']} ---")
        print(f"Pattern: {r['pattern']}")
        print(f"Result: {r['summary']}")
        if r.get("figure"):
            r["figure"].show()


--- Immigration concentrated in competitive districts ---
Pattern: segmentation
Result: Indicators span multiple datasets. Merge needed.

--- QoL diverges from GDP as immigration rises ---
Pattern: divergence
Result: Correlation matrix for hypothesis indicators:
                         median_household_income  foreign_born_pct  real_gdp_millions
median_household_income                    1.000             0.479              0.193
foreign_born_pct                           0.479             1.000              0.657
real_gdp_millions                          0.193             0.657              1.000

--- Immigration precedes QoL decline with lag ---
Pattern: lag_lead
Result: Correlation matrix for hypothesis indicators:
                         foreign_born_pct  median_household_income
foreign_born_pct                    1.000                    0.479
median_household_income             0.479                    1.000

--- Noncitizen populations shift congressional seats ---
Pattern: s

## Ranked Findings Summary

All flags combined and ranked by strength. Review and mark which to
pursue in Layer 3.

In [7]:
from pipeline.flagging import compile_findings_summary

summary = compile_findings_summary(datasets, config,
                                    correlation_threshold=CORRELATION_THRESHOLD,
                                    time_col=TIME_COL,
                                    geo_col=GEO_COL)
display(summary)

,dataset,flag_type,strength,description
rank,,,,
0,state_panel,correlation,strong,real_gdp_millions and naturalizations_total: P...
1,district_crosssection,correlation,strong,"Obama % and McCain %: Pearson r=-0.9865, Spear..."
2,district_crosssection,correlation,strong,"Biden % and Clinton %: Pearson r=0.9845, Spear..."
3,district_crosssection,correlation,strong,2020 Pres. Margin and Trump %.1: Pearson r=-0....
4,district_crosssection,correlation,strong,"Clinton % and Trump %.1: Pearson r=-0.9855, Sp..."
...,...,...,...,...
381,district_crosssection,correlation,moderate,Trump %.1 and No. of Republican Votes: Pearson...
382,district_crosssection,correlation,moderate,cook_pvi_numeric and No. of Republican Votes: ...
383,district_crosssection,correlation,moderate,2020 Pres. Margin and No. of Republican Votes:...
